# Simultaneous Determination of Transmission Expansion Planning & Operation

System planners must simultaneously consider "transmission investment" fixed on a decadal scale, and the subsequent hourly "generation allocation (operation)."
The existing transmission network has small capacity and is chronically congested; depending on which candidate corridors (transmission lines) are expanded, the operational feasibility and costs of multiple demand scenarios (summer peak, normal, low demand) will change. If not expanded, the physical laws (power flow = susceptance $\times$ phase angle difference) themselves do not act on that candidate line — the core difficulty of this problem is the **disjunctive (Big-M) structure** where the construction decision switches the physical laws on or off.

This notebook follows this problem in the following format:

1. **Naive Formulation** — Confirm the scale of the Big-M disjunctive constraints for each candidate line and the DC power flow for each scenario.
2. **Diagnosis** — Observe with `mk.analyze` and see which symptoms trigger.
3. **Looking Inside the Diagnosis** — Directly probe the coefficient scale and symmetry collectors to substantiate the grounds for the triggers.
4. **Trying Improvements** — Actually try the recipe for numerical scale findings (eliminating Big-M) and honestly measure its effectiveness.

Subject: `samples/location_and_network_design/transmission_expansion_operation.py`

In [ ]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" の有無で探すと docs で止まる → pyproject.toml が目印。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/location_and_network_design"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from pyscipopt import Model, quicksum, SCIP_PARAMSETTING

def show(fig):  # 静的サイトに埋め込める形で表示(CDN の plotly.js、requirejsシムを避ける)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import transmission_expansion_operation as tep
from minlpkit.collectors.static_diag import extract_coefficients, scale_summary, residual_scale
from minlpkit.collectors.symmetry import detect_symmetry

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Naive Formulation

`build_model("default")` is on a scale of 9 buses, 8 candidate corridors, and 5 demand scenarios. For each candidate line, 4 disjunctive constraints (`kirchhoff_cand_ub/lb` and `cand_cap_ub/lb`) are duplicated for the number of scenarios, and use a common `BIG_M=400` to represent "if not expanded, power flow = 0 and physical laws are invalidated."

In [ ]:
m = tep.build_model("default")
nB, nE, nCand, nS = m.data["dims"]
n_bin = sum(1 for v in m.getVars() if v.vtype() == "BINARY")
n_cont = sum(1 for v in m.getVars() if v.vtype() != "BINARY")
print(f"バス数={nB} 既存線={nE} 候補線={nCand} シナリオ数={nS}")
print(f"変数: 整数(増強決定) {n_bin} 個 / 連続(潮流・位相角・発電・停電) {n_cont} 個")
print(f"制約: {m.getNConss()} 本(内 disjunctive big-M制約 = 候補線数×シナリオ数×4本 = "
      f"{nCand * nS * 4} 本)")
print(f"共通のBIG_M = {tep.BIG_M}(全候補線・全シナリオで同一の定数)")

The common `BIG_M=400` is the loosest value necessary to "safely zero out the power flow for any candidate line and any scenario," while the absolute value of the power flow that each individual candidate line can actually take (the upper limit of susceptance $\times$ phase angle difference) is much smaller. In Section 4, we will measure how much this "looser M than truly necessary" weakens the LP relaxation.

## 2. Diagnosis (`mk.analyze`)

Actually solve to observe dual bounds, spatial branching, numerical scales, and symmetry, and compare against the diagnostic rules.

In [ ]:
report = mk.analyze(lambda: tep.build_model("default"), name="transmission-default", time_limit=30)
print(report.summary())
print()
print({k: report.metrics.get(k) for k in [
    "gap", "nodes", "nsols", "spatial_share", "has_nonlinear",
    "residual_coef_ratio", "coef_ratio", "residual_bigm_count",
    "n_sym_groups", "largest_sym_group", "sym_sound"]})

`has_nonlinear=False` — This formulation is a **pure MILP** that does not contain true non-convexity (phase angle difference $\times$ susceptance is an affine expression of constant $\times$ variable, not a non-linear term). Even so, SCIP narrows the gap to less than 1% at the first root node. What remains in the diagnosis are two things: **numerical scale** (`numerical_scale`) and **detection of symmetric variable groups** (`symmetry_info`, reference information). In Section 3, we will substantiate the grounds for each trigger.

## 3. Looking Inside the Diagnosis

### 3-1. Numerical Scale — What is Pushing Up `residual_coef_ratio`?

If we look naively at the `residual_coef_ratio` (coefficient ratio after presolve), it shows an extraordinary value of 1e19, but we will check the actual coefficient distribution to see if this is due to a "truly loose Big-M" or another factor.

In [ ]:
m3 = tep.build_model("default")
pre = residual_scale(m3)
print(f"presolve後: min={pre['min']:.3g} max={pre['max']:.3g} ratio={pre['ratio']:.3g} "
      f"median={pre['median']:.3g}")

m4 = tep.build_model("default")
m4.hideOutput()
m4.presolve()  # residual_scale と同じ: presolve後の係数分布を見る
df_coef = extract_coefficients(m4)
df_sorted = df_coef.sort_values("magnitude")
print("--- 最小側(何がminを作っているか) ---")
print(df_sorted.head(5)[["source", "magnitude", "name"]].to_string(index=False))
print("--- 最大側(何がmaxを作っているか) ---")
print(df_sorted.tail(5)[["source", "magnitude", "name"]].to_string(index=False))

In [ ]:
fig = go.Figure(layout=base_layout(
    "presolve後の係数分布(出所別) — 最大は目的係数(計画外停電VOLL)、最小はpresolveの浮動小数点残差",
    "出所", "|係数| (log scale)", height=400))
for src, color in [("制約係数", C["s1"]), ("RHS/LHS", C["muted"]), ("目的係数", C["warn"]),
                   ("変数境界", C["s2"])]:
    sub = df_coef[df_coef["source"] == src]
    if sub.empty:
        continue
    fig.add_trace(go.Box(y=sub["magnitude"], name=src, marker_color=color, boxpoints="outliers"))
fig.update_yaxes(type="log")
show(fig)

The minimum value ($\approx 6.7\text{e-16}$) is a **floating-point residual** that occurred when presolve aggregated constraints (a value that should theoretically be zero but slightly remains due to rounding errors) and is not a "meaningful minimum coefficient" that the model actually has. A meaningful comparison is between the "maximum of the objective coefficients (unplanned outage cost VOLL = 8000 $\times$ scenario weight, making it about 10,000 maximum)" and the "constraint coefficients (susceptance 3-8, capacity 15-55)", and their ratio is roughly on the order of 1e3. **The rule threshold of `residual_coef_ratio >= 1e6` mechanically triggers, but the actual main cause of the ill-conditioning is the numerical residual from presolve, not the looseness of Big-M that should be addressed** — we will confirm this distinction next using the looseness of the Big-M itself.

### 3-2. Symmetry — Grounds for Triggering `symmetry_info`

`detect_symmetry` detects "exchangeable variable groups" using 1-hop signatures.

In [ ]:
groups_df, sym_summary = detect_symmetry(tep.build_model("default"))
print(sym_summary)
print(groups_df.sort_values("size", ascending=False).head(6).to_string(index=False))

In [ ]:
top = groups_df.sort_values("size", ascending=False).head(9)
fig = go.Figure(layout=base_layout(
    "対称(入替可能)な変数群 — 最大群のサイズ", "signature_id(サイズ降順)", "群サイズ", height=360))
fig.add_trace(go.Bar(x=[f"群{i}" for i in top["signature_id"]], y=top["size"],
    marker_color=C["s1"], text=top["size"], textposition="outside", cliponaxis=False))
show(fig)

The contents of the largest group (size 5) consist of the phase angle `theta_b_*` of the same bus lined up for 5 scenarios.
This is not a coincidence — `theta` only appears in the Kirchhoff equalities (power flow = susceptance $\times$ phase angle difference, dependent only on network topology) and does not directly appear in the supply-demand balance equations that depend on demand `demand[s,b]` (what appears there is `flow`/`gen`/`shed`). Therefore, the `theta` of the same bus participates in a set of constraints of exactly the same shape with exactly the same coefficients even if the scenario changes, making them truly exchangeable.
On the other hand, `flow`/`shed` have different `demand` on the right-hand side for each scenario, so they are not symmetric — showing that the boundary of "whether symmetry arises or not" is determined by which constraints (demand-dependent or not) the variables touch. However, as indicated by the severity = good treatment, SCIP automatically handles symmetry with `usesymmetry` enabled by default, so manual lexicographical ordering constraints are usually unnecessary —— the fact that it is actually solved with nodes=1 (root only) substantiates this.

## 4. Trying Improvements — Eliminating Big-M (How Many Times Looser is `BIG_M=400` Than the Truly Necessary Value?)

The `kirchhoff_cand_*` constraints use a common `BIG_M=400`, but the absolute value of the power flow that a candidate line `c` can actually take is bounded by `b_cand[c] * (full width of θ range)` (since the $\theta$ range is $\pm 0.5$ rad, the full width is 1.0).
We create a version replacing this with a tight `M_c = b_cand[c] * 1.0` for each candidate line, and honestly compare both (a) actual measurement under default settings (with presolve/cut) and (b) the quality of the formulation itself (pure LP relaxation with presolve/cut turned off).

In [ ]:
def build_tight(scale="default"):
    # 候補線ごとのタイトなBig-M(= b_cand[c] * theta全幅)に置き換えた版
    d = tep._data(scale)
    nB, nE, nC, nS = d["nB"], d["nE"], d["nC"], d["nS"]
    existing, cand_edges = d["existing"], d["cand_edges"]
    b_exist, cap_exist = d["b_exist"], d["cap_exist"]
    b_cand, cap_cand, invest_cost = d["b_cand"], d["cap_cand"], d["invest_cost"]
    gen_bus, gen_cap, gen_cost = d["gen_bus"], d["gen_cap"], d["gen_cost"]
    demand, scenario_weight = d["demand"], d["scenario_weight"]
    THETA_MAX = 0.5
    tight_M = b_cand * (2 * THETA_MAX)  # 候補線ごとのタイトなBig-M

    m = Model("Transmission_Expansion_Operation_TightM")
    B, E, Cs, S = range(nB), range(nE), range(nC), range(nS)
    G = range(len(gen_bus))
    REF = 0
    build = {c: m.addVar(vtype="B", name=f"build_{c}") for c in Cs}
    theta, flow_e, flow_c, gen, shed = {}, {}, {}, {}, {}
    for s in S:
        for b_ in B:
            lb = 0.0 if b_ == REF else -THETA_MAX
            ub = 0.0 if b_ == REF else THETA_MAX
            theta[b_, s] = m.addVar(lb=lb, ub=ub, name=f"theta_{b_}_{s}")
        for e in E:
            flow_e[e, s] = m.addVar(lb=-float(cap_exist[e]), ub=float(cap_exist[e]), name=f"flow_e_{e}_{s}")
        for c in Cs:
            flow_c[c, s] = m.addVar(lb=-float(cap_cand[c]), ub=float(cap_cand[c]), name=f"flow_c_{c}_{s}")
        for g in G:
            gen[g, s] = m.addVar(lb=0.0, ub=float(gen_cap[g]), name=f"gen_{g}_{s}")
        for b_ in B:
            shed[b_, s] = m.addVar(lb=0.0, ub=float(demand[s, b_]), name=f"shed_{b_}_{s}")
    for s in S:
        for e, (i, j) in enumerate(existing):
            m.addCons(flow_e[e, s] == float(b_exist[e]) * (theta[i, s] - theta[j, s]))
        for c, (i, j) in enumerate(cand_edges):
            m.addCons(flow_c[c, s] <= float(cap_cand[c]) * build[c])
            m.addCons(flow_c[c, s] >= -float(cap_cand[c]) * build[c])
            Mc = float(tight_M[c])
            m.addCons(flow_c[c, s] - float(b_cand[c]) * (theta[i, s] - theta[j, s]) <= Mc * (1 - build[c]))
            m.addCons(flow_c[c, s] - float(b_cand[c]) * (theta[i, s] - theta[j, s]) >= -Mc * (1 - build[c]))
        for b_ in B:
            inflow = quicksum(flow_e[e, s] for e, (i, j) in enumerate(existing) if j == b_) \
                - quicksum(flow_e[e, s] for e, (i, j) in enumerate(existing) if i == b_) \
                + quicksum(flow_c[c, s] for c, (i, j) in enumerate(cand_edges) if j == b_) \
                - quicksum(flow_c[c, s] for c, (i, j) in enumerate(cand_edges) if i == b_)
            local_gen = quicksum(gen[g, s] for g in G if int(gen_bus[g]) == b_)
            m.addCons(local_gen + inflow + shed[b_, s] == float(demand[s, b_]))
    invest = quicksum(float(invest_cost[c]) * build[c] for c in Cs)
    operation = quicksum(float(scenario_weight[s]) * (
        quicksum(float(gen_cost[g]) * gen[g, s] for g in G)
        + quicksum(tep.VOLL * shed[b_, s] for b_ in B)) for s in S)
    m.setObjective(invest + operation, "minimize")
    m.data = dict(build=build)
    return m

print("loose M(共通400)   :", tep.BIG_M)
tight_vals = tep._data("default")["b_cand"] * 1.0
print("tight M(候補線ごと) :", np.round(tight_vals, 2), "-- 最大でも", round(float(tight_vals.max()), 2))

In [ ]:
df = mk.compare_variants(
    {"loose-M(400共通)": lambda: tep.build_model("default"),
     "tight-M(候補線ごと)": build_tight}, time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

Under default settings (with presolve/cut), the `root_dual` / `final_gap` / `nodes` for both are almost identical. Because SCIP's presolve automatically tightens Big-M within obvious ranges, the 17- to 50-fold tightening from "400 $\rightarrow$ maximum 8.0 per candidate line" has almost no effect in default operation (the speed of nodes=1 seen in Section 2 is also because presolve has already created a practically tight relaxation).
The difference in the quality of the formulation itself only appears in the pure LP relaxation with presolve/cut turned off.

In [ ]:
def root_lp_bound(build_fn):
    m = build_fn()
    m.hideOutput()
    m.setPresolve(SCIP_PARAMSETTING.OFF)
    m.setSeparating(SCIP_PARAMSETTING.OFF)
    m.setHeuristics(SCIP_PARAMSETTING.OFF)
    m.setParam("limits/nodes", 1)
    m.optimize()
    return m.getDualbound()

loose_lp = root_lp_bound(lambda: tep.build_model("default"))
tight_lp = root_lp_bound(build_tight)
print(f"純粋LP緩和(presolve/cut/heuristics off): loose-M={loose_lp:.1f}  tight-M={tight_lp:.1f}  "
      f"差={(tight_lp - loose_lp):+.1f}({(tight_lp - loose_lp) / abs(loose_lp) * 100:+.1f}%)")

In [ ]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.15,
    subplot_titles=("既定設定のroot_dual(presolve/cutあり) — ほぼ差なし",
                    "純粋LP緩和(presolve/cut/heuristics off) — 定式化の質の差"))
loose_row = df.loc[df["variant"] == "loose-M(400共通)"].iloc[0]
tight_row = df.loc[df["variant"] == "tight-M(候補線ごと)"].iloc[0]
fig.add_trace(go.Bar(x=["loose-M", "tight-M"], y=[loose_row["root_dual"], tight_row["root_dual"]],
    marker_color=[C["muted"], C["s1"]], text=[f"{v:.0f}" for v in
    [loose_row["root_dual"], tight_row["root_dual"]]], textposition="outside", cliponaxis=False,
    showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=["loose-M", "tight-M"], y=[loose_lp, tight_lp],
    marker_color=[C["muted"], C["s1"]], text=[f"{v:.0f}" for v in [loose_lp, tight_lp]],
    textposition="outside", cliponaxis=False, showlegend=False), row=1, col=2)
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=380,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="Big-M排除の効果 — 既定設定 vs 純粋LP緩和", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
show(fig)

## Summary

- The essential difficulty of this problem lies in the investment selection of simultaneously satisfying "whether candidate line expansion (integer) creates a physical path to generation sites cheaper than existing ones" across all 5 demand scenarios (two-stage robust planning). It is a pure MILP containing no true non-convexity, but due to the disjunctive (Big-M) structure, the integer decisions alter the feasible region of continuous variables itself.
- Because SCIP's presolve and built-in symmetry handling are powerful, at this scale, the gap and node count themselves do not become subjects of diagnosis (gap < 1% at root 1 node). **What should be picked up in diagnosis is distinguishing between "what presolve tightens for us" and "what is essentially necessary"**, and we showed in Section 3 that mechanical threshold checks like `residual_coef_ratio` can erroneously report numerical residuals (floating-point errors) from presolve as "ill-conditioning".
- The Big-M itself had room for a 50-fold tightening from `400` to a maximum of `8.0` per candidate line, but it had almost no effect in actual measurements under default settings (presolve does practically the same work).
  The effect can only be observed in a pure LP relaxation with presolve/cut/heuristics turned off — a classic example of "when Big-M is ineffective".

### Business Decisions Corresponding to this Diagnosis

The proposal of "which candidate corridors should be expanded" submitted by system planners to the investment committee is easy for a single scenario, but becomes NP-hard the moment trade-offs between multiple scenarios arise.
The diagnosis in this notebook provides practical judgment material for the planning work itself: "At this scale, SCIP's default settings solve it fast enough, so the return on investment for complex preprocessing (manual Big-M elimination) is low."

Related: [Method Guide 3. Big-M/Indicator](../../playbook/03-bigm.md) /
[08. Condition Number and Numerical Soundness](../../playbook/08-condition.md) /
Sample Catalog [Location and Network Design](../../samples/location_and_network_design.md)